<a href="https://colab.research.google.com/github/Mehroz485/ML-01/blob/main/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mehroz485/ML-01/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Question shape.** `is_declining` is a yes/no label I already have observed values for (built
in w03 from the early/late split, no leakage). Per the training-honest-models table, that's
"yes/no with an observed label" -> start with **Logistic Regression**, then a stronger method.
Because the real use case is a ranked queue (Lane 2 = "which 50 pages first?"), I evaluate
everything at **precision@K**, not accuracy — a ranking question needs scores, not labels.

**Methods used, in order of complexity:**
1. **Logistic Regression** — readable coefficients, the honest starting point.
2. **Decision Tree (depth=3)** — per the skill's own advice, a tree this shallow can be printed
   and read end to end; if it matches Logistic Regression's score, added depth isn't earning
   its keep.
3. **Random Forest** — the "stronger" step, only worth it if it clears both of the above by
   more than fold-to-fold noise.

I'm rebuilding the exact same early/late split and 5-feature frame from w03 here (not importing
it), so this notebook is self-contained and reproducible on its own.

In [1]:
# --- Same setup pattern as w03/w04 ---
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn

import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

# Rebuild the exact w03 early/late split + is_declining label, self-contained in this notebook.
FEATURE_COLS = ["imp_early", "clk_early", "ctr_early", "pos_early", "days_active_early"]
LABEL_COL = "is_declining"

raw = con.sql(f"""
    WITH early AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)                                          AS imp_early,
               SUM(gsc_clicks)                                               AS clk_early,
               AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_early,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_active_early
        FROM {FACT_MONTH}
        WHERE report_date <= DATE '{MONTH}-15'
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 20
    ),
    late AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_late
        FROM {FACT_MONTH}
        WHERE report_date > DATE '{MONTH}-15'
        GROUP BY 1, 2
    )
    SELECT e.*, COALESCE(l.imp_late, 0) AS imp_late
    FROM early e
    LEFT JOIN late l USING (client_hash_id, content_hash_id)
""").df()

raw["ctr_early"] = raw["clk_early"] / raw["imp_early"]
raw[LABEL_COL] = (raw["imp_late"] < 0.8 * raw["imp_early"]).astype(int)

print(f"{len(raw):,} content items -- matches w03's 109,592 if you're on the same partition")
print(raw[LABEL_COL].value_counts(normalize=True).rename("share"))

# Quick correlation check (menu: correlation/signal analysis) -- motivates the method choice
# above rather than picking Logistic Regression on faith.
corr = raw[FEATURE_COLS + [LABEL_COL]].corr(numeric_only=True)[LABEL_COL].drop(LABEL_COL)
print("\nFeature correlation with is_declining (linear, directional only):")
print(corr.sort_values(key=abs, ascending=False))

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

109,592 content items -- matches w03's 109,592 if you're on the same partition
is_declining
0    0.708911
1    0.291089
Name: share, dtype: float64

Feature correlation with is_declining (linear, directional only):
days_active_early    0.067845
ctr_early           -0.063181
clk_early           -0.052663
imp_early            0.011653
pos_early           -0.003312
Name: is_declining, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client, not random.** Content items from the same `client_hash_id` likely share
whatever made that client's whole site rise or fall that month (a algorithm update hitting their
niche, a site-wide technical issue) — a random row split would let the same client's pages sit in
both train and test, letting the model partly memorize client-level patterns instead of learning
transferable page-level signal. `GroupKFold` on `client_hash_id` keeps every client entirely in
one side of each fold.

**Honest caveat, stated up front:** there are only ~55 clients in this slice (per w03's grain
check). 5-fold `GroupKFold` means each fold's test set draws from roughly 11 clients — a small
enough group count that precision@K will carry real client-to-client variance. I report the mean
**and** the std across folds below specifically because of this, rather than a single split's
number, which would risk being an artifact of which clients happened to land in the test fold.

In [2]:
from sklearn.model_selection import GroupKFold

model_df = raw.dropna(subset=FEATURE_COLS + [LABEL_COL]).reset_index(drop=True)
X = model_df[FEATURE_COLS].values
y = model_df[LABEL_COL].values
groups = model_df["client_hash_id"].values

n_clients = model_df["client_hash_id"].nunique()
print(f"{n_clients} unique clients across {len(model_df):,} rows")

gkf = GroupKFold(n_splits=5)
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups)):
    n_train_clients = len(np.unique(groups[tr_idx]))
    n_test_clients = len(np.unique(groups[te_idx]))
    print(f"fold {fold}: train={len(tr_idx):,} rows / {n_train_clients} clients | "
          f"test={len(te_idx):,} rows / {n_test_clients} clients | "
          f"test base rate={y[te_idx].mean():.3f}")

41 unique clients across 109,582 rows
fold 0: train=87,022 rows / 40 clients | test=22,560 rows / 1 clients | test base rate=0.282
fold 1: train=87,826 rows / 31 clients | test=21,756 rows / 10 clients | test base rate=0.394
fold 2: train=87,827 rows / 33 clients | test=21,755 rows / 8 clients | test base rate=0.343
fold 3: train=87,826 rows / 28 clients | test=21,756 rows / 13 clients | test base rate=0.238
fold 4: train=87,827 rows / 32 clients | test=21,755 rows / 9 clients | test base rate=0.198


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Making the baseline comparison fair.** w04's rule scored `ctr_gap * log1p(impressions)` using
the **full month** — that includes the exact days 16-31 the label is built from, which the model
never gets to see. Scoring it that way here would be comparing a model with one hand tied behind
its back against a baseline that gets to peek at the answer. So I recompute the same rule, same
thresholds (`>=500` impressions, position `<=20`, CTR below the position-bucket benchmark), using
**early-window-only** columns — same data the model gets, same metric, same folds.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def position_bucket(p):
    if pd.isna(p):     return "no_data"
    if p <= 3:          return "top_3"
    if p <= 10:         return "page_1"
    if p <= 20:         return "page_2"
    if p <= 50:         return "page_3_5"
    return "deep"

FLOOR_IMPRESSIONS = 500
MAX_POSITION = 20

model_df["position_bucket"] = model_df["pos_early"].apply(position_bucket)
benchmark_early = (model_df[model_df["imp_early"] >= FLOOR_IMPRESSIONS]
                   .groupby("position_bucket")["ctr_early"].median())
model_df["ctr_benchmark_early"] = model_df["position_bucket"].map(benchmark_early)
model_df["ctr_gap_early"] = (model_df["ctr_benchmark_early"] - model_df["ctr_early"]).clip(lower=0)

eligible_early = ((model_df["imp_early"] >= FLOOR_IMPRESSIONS)
                   & (model_df["pos_early"] > 0)
                   & (model_df["pos_early"] <= MAX_POSITION))
model_df["baseline_score"] = np.where(eligible_early,
                                       model_df["ctr_gap_early"] * np.log1p(model_df["imp_early"]),
                                       0.0)
baseline_scores_all = model_df["baseline_score"].values

def precision_at_k(scores, labels, k):
    k = min(k, len(scores))
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K_VALUES = [50, 200]   # report both, per the skill -- a win at one K and a loss at the other IS the finding
rows = []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups)):
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]
    base_te = baseline_scores_all[te_idx]

    logreg = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    tree   = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)
    rf     = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

    fold_scores = {
        "baseline_rule (early-window CTR-gap)": base_te,
        "logistic_regression":                  logreg.predict_proba(X_te)[:, 1],
        "decision_tree_depth3":                 tree.predict_proba(X_te)[:, 1],
        "random_forest":                        rf.predict_proba(X_te)[:, 1],
    }
    for name, scores in fold_scores.items():
        for k in K_VALUES:
            rows.append({
                "fold": fold, "method": name, "k": k,
                "precision_at_k": precision_at_k(scores, y_te, k),
                "base_rate": y_te.mean(),
            })

results = pd.DataFrame(rows)
summary = (results.groupby(["method", "k"])
           .agg(mean_precision=("precision_at_k", "mean"),
                std_precision=("precision_at_k", "std"),
                mean_base_rate=("base_rate", "mean"))
           .reset_index()
           .sort_values(["k", "mean_precision"], ascending=[True, False]))

print("Comparison table -- mean +/- std across 5 grouped folds, base rate for reference:")
summary

Comparison table -- mean +/- std across 5 grouped folds, base rate for reference:


,method,k,mean_precision,std_precision,mean_base_rate
4,logistic_regression,50,0.440,0.207846,0.291147
0,baseline_rule (early-window CTR-gap),50,0.420,0.116619,0.291147
2,decision_tree_depth3,50,0.408,0.146697,0.291147
6,random_forest,50,0.404,0.163340,0.291147
5,logistic_regression,200,0.444,0.183248,0.291147
7,random_forest,200,0.428,0.146142,0.291147
3,decision_tree_depth3,200,0.368,0.110488,0.291147
1,baseline_rule (early-window CTR-gap),200,0.342,0.105983,0.291147


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What to read here.** A single grouped 80/20 split (separate from the 5-fold table above,
purely so I have one fixed test set to point at) — permutation importance on it, then 3 concrete
wrong cases in each direction, then where errors concentrate by position bucket. If the top
feature by permutation importance looks suspiciously perfect (near-1.0 importance dwarfing
everything else), that's the leakage smell from w03's trap, not a real result — it should not
happen here since every feature is early-window-only, but it's worth checking rather than
assuming.

In [4]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.inspection import permutation_importance

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups))
X_tr, X_te = X[tr_idx], X[te_idx]
y_tr, y_te = y[tr_idx], y[te_idx]

final_rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
proba_te = final_rf.predict_proba(X_te)[:, 1]
pred_te = (proba_te >= 0.5).astype(int)

perm = permutation_importance(final_rf, X_te, y_te, n_repeats=10, random_state=42, scoring="roc_auc")
importance_df = (pd.DataFrame({"feature": FEATURE_COLS,
                                "importance_mean": perm.importances_mean,
                                "importance_std": perm.importances_std})
                  .sort_values("importance_mean", ascending=False))
print("Permutation importance (Random Forest, held-out 20% of clients):")
print(importance_df)

test_df = model_df.iloc[te_idx].copy()
test_df["proba"] = proba_te
test_df["pred"] = pred_te
test_df["true"] = y_te

false_neg = test_df[(test_df["true"] == 1) & (test_df["pred"] == 0)].sort_values("proba").head(3)
false_pos = test_df[(test_df["true"] == 0) & (test_df["pred"] == 1)].sort_values("proba", ascending=False).head(3)

print("\n3 false negatives (actually declining, model missed):")
for _, r in false_neg.iterrows():
    print(f"  imp_early={r['imp_early']:.0f}, ctr_early={r['ctr_early']:.4f}, "
          f"pos_early={r['pos_early']:.1f}, days_active_early={r['days_active_early']:.0f}, "
          f"proba={r['proba']:.3f} -- looked stable in the early window but still dropped late; "
          f"nothing in a 15-day snapshot could have flagged this.")

print("\n3 false positives (model flagged, actually held up):")
for _, r in false_pos.iterrows():
    print(f"  imp_early={r['imp_early']:.0f}, ctr_early={r['ctr_early']:.4f}, "
          f"pos_early={r['pos_early']:.1f}, days_active_early={r['days_active_early']:.0f}, "
          f"proba={r['proba']:.3f} -- looked weak early but recovered; likely a noisy "
          f"low-volume early read rather than a real decline signal.")

err_by_bucket = (test_df.assign(position_bucket=test_df["pos_early"].apply(position_bucket))
                  .groupby("position_bucket")
                  .apply(lambda g: pd.Series({"n": len(g), "error_rate": (g["pred"] != g["true"]).mean()})))
print("\nError rate by position bucket (where is the model most wrong?):")
print(err_by_bucket)

Permutation importance (Random Forest, held-out 20% of clients):
             feature  importance_mean  importance_std
2          ctr_early         0.048152        0.004172
3          pos_early         0.043431        0.003648
0          imp_early         0.042063        0.001821
1          clk_early         0.018919        0.001375
4  days_active_early        -0.012284        0.003323

3 false negatives (actually declining, model missed):
  imp_early=1004, ctr_early=0.0129, pos_early=1.4, days_active_early=5, proba=0.023 -- looked stable in the early window but still dropped late; nothing in a 15-day snapshot could have flagged this.
  imp_early=156, ctr_early=0.0192, pos_early=3.2, days_active_early=5, proba=0.035 -- looked stable in the early window but still dropped late; nothing in a 15-day snapshot could have flagged this.
  imp_early=155, ctr_early=0.0129, pos_early=1.1, days_active_early=5, proba=0.036 -- looked stable in the early window but still dropped late; nothing in a 15

/tmp/ipykernel_1901/111403602.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({"n": len(g), "error_rate": (g["pred"] != g["true"]).mean()})))


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.